# Crouzeix's Conjecture

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order






@njit
def evaluate_poly_at_point_njit(z, coeffs):
  """Evaluates a polynomial at a complex point z using Horner's method.

  `coeffs` must be a NumPy array.
  """
  # Start with the coefficient of the highest power
  res = coeffs[-1]
  # Iterate from the second to last coefficient down to the first
  for i in range(len(coeffs) - 2, -1, -1):
    res = res * z + coeffs[i]
  return res


@njit
def evaluate_polynomial_matrix_njit(matrix_A, poly_coeffs):
  """Evaluates a polynomial at a matrix. `poly_coeffs` must be a NumPy array."""
  result = np.zeros_like(matrix_A, dtype=matrix_A.dtype)
  power_A = np.eye(matrix_A.shape[0], dtype=matrix_A.dtype)
  for coeff in poly_coeffs:
    result += coeff * power_A
    power_A = power_A @ matrix_A
  return result


@njit
def maximize_polynomial_field_of_values_njit(
    poly_coeffs, A, num_theta_points=1000
):
  """Maximizes |p(<Ax,x>)|. `poly_coeffs` must be a NumPy array."""
  max_p_value = -np.inf
  theta_vals = np.linspace(0, 2 * np.pi, num_theta_points)

  for theta in theta_vals:
    H_theta = 0.5 * (np.exp(1j * theta) * A + np.exp(-1j * theta) * A.conj().T)
    eigenvalues, eigenvectors = np.linalg.eig(H_theta)

    largest_eigenvalue_index = np.argmax(eigenvalues.real)

    v_theta = eigenvectors[:, largest_eigenvalue_index]
    v_theta = v_theta / np.linalg.norm(v_theta)
    z_theta = v_theta.conj().T @ A @ v_theta

    # Call the jitted polynomial evaluator with the complex point and NumPy array of coeffs
    current_p_value = np.abs(evaluate_poly_at_point_njit(z_theta, poly_coeffs))

    if current_p_value > max_p_value:
      max_p_value = current_p_value

  return max_p_value


# ===================================================================
# Wrapper Function
# This function prepares the data and calls the fast Numba code.
# ===================================================================


def evaluate_construction(A, p_coeffs, num_theta_points=1000):
  """Wrapper function that calls the high-performance Numba-jitted core."""
  # **FIX**: Convert the list of coefficients to a NumPy array here.
  # This array (not a Polynomial object) is then passed to the jitted functions.
  p_coeffs_arr = np.array(p_coeffs, dtype=np.complex128)

  # if the matrix is 2x2 or smaller, return -np.inf
  if A.shape[0] <= 9:
    return -np.inf

  # --- Call the Numba-jitted functions with the correct array type ---
  try:
    p_A = evaluate_polynomial_matrix_njit(A, p_coeffs_arr)
  except Exception as e:
    print(f'Error in evaluate_polynomial_matrix_njit: {e}')
    return -np.inf
  max_val_fov = maximize_polynomial_field_of_values_njit(
      p_coeffs_arr, A, num_theta_points
  )

  norm_p_A_2 = la.norm(p_A, ord=2)

  final_score = norm_p_A_2 / max_val_fov
  return final_score


##### Hidden variants of above code


def evaluate_polynomial_matrix_h(matrix_a, poly_coeffs):
  """Evaluates a polynomial at a matrix.

  Args:
      matrix_a: NumPy array, the matrix A.
      poly_coeffs: List of polynomial coefficients in increasing order of
        degree.

  Returns:
      NumPy array, p(A).
  """
  result = np.zeros_like(matrix_a, dtype=np.complex128)
  power_a = np.eye(matrix_a.shape[0])  # Identity matrix
  for coeff in poly_coeffs:
    result += coeff * power_a
    power_a = power_a @ matrix_a  # Next power of A
  return result


def maximize_polynomial_field_of_values_h(p, matrix_a, num_theta_points=1000):
  """Field of Values method - Maximizing MAGNITUDE of p(<Ax,x>) using the theorem."""
  max_p_value = -float('inf')
  optimal_z_theta = None

  for theta in np.linspace(0, 2 * np.pi, num_theta_points):
    h_theta = 0.5 * (
        np.exp(1j * theta) * matrix_a + np.exp(-1j * theta) * matrix_a.conj().T
    )
    eigenvalues, eigenvectors = np.linalg.eig(h_theta)
    largest_eigenvalue_index = np.argmax(eigenvalues)
    v_theta = eigenvectors[:, largest_eigenvalue_index]
    v_theta = v_theta.astype(np.complex128)
    v_theta = v_theta / np.linalg.norm(v_theta)
    z_theta = v_theta.conj().T @ matrix_a @ v_theta
    current_p_value = np.abs(p(z_theta))
    if current_p_value > max_p_value:
      max_p_value = current_p_value
      optimal_z_theta = z_theta
  return max_p_value, optimal_z_theta


def evaluate_construction_h(matrix_a, p_coeffs, num_theta_points=1000):
  p_poly = poly.Polynomial(p_coeffs)
  p_of_a = evaluate_polynomial_matrix_h(matrix_a, p_coeffs)
  norm_p_of_a_2 = la.norm(p_of_a, ord=2)
  max_val_fov, _ = maximize_polynomial_field_of_values_h(
      p_poly, matrix_a, num_theta_points
  )
  return norm_p_of_a_2 / max_val_fov


##########


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation in code."""
  formatted_feedback = {}
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(
    hypers: Mapping[str, Any],
) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the numerical bound for the polygons if valid, or 0 if invalid."""
  result = {}
  feedback = {}
  del hypers
  best_construction = search_for_best_matrix_and_poly()
  result['score'] = evaluate_construction(
      best_construction[0], best_construction[1], num_theta_points=10000
  )

  feedback['best_matrix'] = best_construction[0]
  feedback['best_poly'] = best_construction[1]
  feedback['best_score_found'] = result['score']
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""Finds a function that give the best bound for Bogdan's first problem."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import re
from typing import Any, Callable, Mapping
import scipy.linalg as la
import numpy.polynomial.polynomial as poly
import numba

njit = numba.njit
minimize = optimize.minimize



def search_for_best_matrix_and_poly() -> tuple[np.ndarray, list[float]]:
  """Function to search for the best construction."""
  variable_name = 'best_matrix_iqhd'
  if variable_name in globals():
    matrix_a = globals()[variable_name]
  else:
    matrix_a = np.eye(15, dtype=np.complex128)

  variable_name = 'best_poly_iqhd'
  if variable_name in globals():
    p_coeffs = globals()[variable_name]
  else:
    p_coeffs = [1, 2, 3 - 1j]
  best_matrix = matrix_a.copy()
  best_poly_coefficients = p_coeffs.copy()
  best_score = evaluate_construction(matrix_a, p_coeffs, num_theta_points=1000)
  start_time = time.time()
  random_runtime = np.random.randint(100, 1000)
  num_evals = 0
  while time.time() - start_time < random_runtime:
    num_evals += 1
    p_coeffs[np.random.randint(2)] += np.random.uniform(-1, 1)
    matrix_a[np.random.randint(2), np.random.randint(2)] += np.random.uniform(
        -1, 1
    )
    score = evaluate_construction(matrix_a, p_coeffs, num_theta_points=1000)
    if score > best_score:
      best_score = score
      best_matrix = matrix_a.copy()
      best_poly_coefficients = p_coeffs.copy()
      logging.info(score)
  logging.info(num_evals)
  return best_matrix, best_poly_coefficients

**Prompt used**

Act as an expert in numerical analysis, computational mathematics, and software optimization. Your primary area of expertise is the relationship between matrices and polynomials, specifically concerning matrix norms and the field of values (numerical range).

Your task is to write a Python search function, search_for_best_matrix_and_poly(), that finds an optimal square complex matrix A and a list of complex polynomial coefficients p_coeffs.
The goal is to find the pair (A, p_coeffs) that maximizes the score returned by the following evaluation function (you have access to this function and do not need to reimplement it):

def evaluate_construction(A: np.ndarray, p_coeffs: list[complex], num_theta_points: int = 1000) -> float:
    """
    Evaluates the quality of a matrix A and polynomial p.
    A higher score is better.

The score is the ratio:
    norm(p(A)) / max(|p(z)| for z in the numerical range of A)
"""
# ... implementation details ...
Understanding the Score:
The core of the problem is to maximize this ratio. This creates a tension you must exploit:

The Numerator (norm(p(A))): This is the 2-norm (the largest singular value) of the matrix that results from evaluating the polynomial p at the matrix A. You want this to be as large as possible.

The Denominator (max(|p(z)|)): This is the maximum absolute value of the polynomial p when evaluated at any complex number z inside the field of values (or numerical range) of A. You want this to be as small as possible.

Your challenge is to find a matrix A and polynomial p where p(A) is large, while the values of p(z) remain small across the entire numerical range of A. The num_theta_points parameter determines how accurate the score computation is, higher numbers yield more accurate answers.

Your search algorithm will have at most 1000 seconds to run, after which it has to return the best construction it has found. This will then be evaluated by a team of experts to a very high precision. The matrix you construct has to be a square matrix, but it can have arbitrary size otherwise (but at least 10x10).

In [ ]:
#@title Code evolved by AlphaEvolve



def search_for_best_matrix_and_poly() -> tuple[np.ndarray, list[float]]:
  """Function to search for the best construction."""
  # CRAZY IDEA: Start with a block nilpotent matrix A = [[0, 2*I], [0, 0]]
  # and polynomial p(z)=z. This construction has a theoretical score of 2,
  # providing a stronger baseline than the Jordan block.
  # A is nilpotent (A^2=0), its numerical range W(A) is the unit disk.
  # For p(z)=z, norm(p(A))=norm(A)=2, and max|p(z)| on W(A) is 1.
  n = 20 # Using a larger, even-sized matrix. Must be >= 10.
  half_n = n // 2
  M = 2.0 * np.eye(half_n, dtype=np.complex128)
  initial_matrix = np.block([
      [np.zeros((half_n, half_n)), M],
      [np.zeros((half_n, half_n)), np.zeros((half_n, half_n))]
  ]).astype(np.complex128)

  # The corresponding polynomial p(z) = z
  initial_poly_coeffs = [0.0] * n
  initial_poly_coeffs[1] = 1.0 + 0.0j

  # Evaluate the new construction
  initial_score = evaluate_construction(initial_matrix, initial_poly_coeffs, num_theta_points=1000)

  # Check if a better solution is available globally and use it if it's superior
  global_matrix = globals().get('best_matrix_iqhd', None)
  global_poly_coeffs = globals().get('best_poly_iqhd', None)

  best_matrix = initial_matrix.copy()
  best_poly_coefficients = initial_poly_coeffs.copy()
  best_score = initial_score

  if global_matrix is not None and global_poly_coeffs is not None:
    try:
      global_score = evaluate_construction(global_matrix, global_poly_coeffs, num_theta_points=1000)
      if global_score > best_score:
        best_matrix = global_matrix.copy()
        best_poly_coefficients = global_poly_coeffs.copy()
        best_score = global_score
    except Exception as e:
      logging.warning(f"Error evaluating global best construction: {e}. Sticking to Jordan construction.")

  # Use the chosen initial best for the search
  matrix_a = best_matrix.copy()
  p_coeffs = best_poly_coefficients.copy()
  best_score = evaluate_construction(matrix_a, p_coeffs, num_theta_points=1000)
  start_time = time.time()
  random_runtime = np.random.randint(100, 1000)
  # Store current matrix and polynomial for potential rollback
  current_matrix = matrix_a.copy()
  current_p_coeffs = p_coeffs.copy()

  num_evals = 0
  max_runtime = 990 # Leave a small buffer for final evaluation and logging

  # Adapt the perturbation magnitude over time or based on score changes
  perturb_scale_matrix = 0.05 # Initial small scale
  perturb_scale_poly = 0.05

  # Track number of consecutive failures to adapt perturbation
  failures_matrix = 0
  failures_poly = 0
  max_failures = 50 # After this many failures, adjust scale

  # Log the initial score to track progress
  logging.info(f"Initial score: {best_score:.4f}")

  while time.time() - start_time < max_runtime:
    num_evals += 1

    # Decide whether to perturb matrix or polynomial
    if np.random.rand() < 0.5: # Perturb matrix
      # Choose a random element to perturb
      r, c = np.random.randint(current_matrix.shape[0], size=2)

      # Store old value for rollback
      old_val = current_matrix[r, c]

      # Apply complex perturbation
      current_matrix[r, c] += perturb_scale_matrix * (np.random.uniform(-1, 1) + 1j * np.random.uniform(-1, 1))

      score = evaluate_construction(current_matrix, current_p_coeffs, num_theta_points=1000)

      if score > best_score:
        best_score = score
        best_matrix = current_matrix.copy()
        best_poly_coefficients = current_p_coeffs.copy()
        logging.info(f"New best score (matrix perturb): {best_score:.4f}")
        failures_matrix = 0 # Reset failures on success
        # Optionally increase perturb_scale_matrix for more aggressive search
        # perturb_scale_matrix = min(0.1, perturb_scale_matrix * 1.1)
      else:
        # Revert changes if score didn't improve
        current_matrix[r, c] = old_val
        failures_matrix += 1
        if failures_matrix > max_failures:
            perturb_scale_matrix = max(0.001, perturb_scale_matrix * 0.9) # Decrease scale
            failures_matrix = 0 # Reset counter

    else: # Perturb polynomial coefficients
      # Choose a random coefficient to perturb
      idx = np.random.randint(len(current_p_coeffs))

      # Store old value for rollback
      old_coeff = current_p_coeffs[idx]

      # Apply complex perturbation
      current_p_coeffs[idx] += perturb_scale_poly * (np.random.uniform(-1, 1) + 1j * np.random.uniform(-1, 1))

      score = evaluate_construction(current_matrix, current_p_coeffs, num_theta_points=1000)

      if score > best_score:
        best_score = score
        best_matrix = current_matrix.copy()
        best_poly_coefficients = current_p_coeffs.copy()
        logging.info(f"New best score (poly perturb): {best_score:.4f}")
        failures_poly = 0 # Reset failures on success
        # Optionally increase perturb_scale_poly
        # perturb_scale_poly = min(0.1, perturb_scale_poly * 1.1)
      else:
        # Revert changes if score didn't improve
        current_p_coeffs[idx] = old_coeff
        failures_poly += 1
        if failures_poly > max_failures:
            perturb_scale_poly = max(0.001, perturb_scale_poly * 0.9) # Decrease scale
            failures_poly = 0 # Reset counter

  logging.info(f"Search finished. Total evaluations: {num_evals}. Best score found: {best_score:.4f}")
  return best_matrix, best_poly_coefficients


## What AlphaEvolve found

AlphaEvolve was asked to find an example improving the lower bound of $C = 2$ by optimizing the ratio $\|p(A)\|_{op} / \sup_{z \in W(A)} |p(z)|$ over matrices $A$ and polynomials $p$. It was tested with matrices of variable sizes but did not find any examples that could go beyond matching the literature bound of 2.